<a href="https://colab.research.google.com/github/ahmadtariq2004/Evolvix-Ai-ML-Internship-ahmadtariq2004/blob/main/Week_2/Task_2/WEEK_2_TASK_2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Load the dataset
try:
    df = pd.read_csv('cars.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: cars.csv not found. Please ensure the file is in the correct directory.")
    # In a real scenario, you might want to exit or provide a mock DataFrame.
    # For this exercise, assume the file exists.
    raise # Re-raise the error to stop execution if file is missing.

# Display initial info
print("\nInitial DataFrame Info:")
df.info()
print("\nInitial DataFrame Head:")
print(df.head())

# Assume 'MSRP' is the target variable based on typical car analysis tasks
if 'MSRP' not in df.columns:
    print("Error: 'MSRP' column not found. Please specify the target variable or adjust the code.")
    # Attempt to find another suitable target or raise an error
    numeric_cols = df.select_dtypes(include=np.number).columns
    if 'Price' in numeric_cols: # Common alternative
        target_col = 'Price'
    elif not numeric_cols.empty: # Pick first numeric if no 'MSRP' or 'Price'
        target_col = numeric_cols[0]
        print(f"Using '{target_col}' as target variable instead of 'MSRP'.")
    else:
        raise ValueError("No suitable numeric column found for target variable.")
    y = df[target_col]
    X = df.drop(target_col, axis=1)
else:
    target_col = 'MSRP'
    X = df.drop(target_col, axis=1)
    y = df[target_col]

# Drop rows where the target variable is missing
df.dropna(subset=[target_col], inplace=True)

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Drop columns that are usually identifiers or have too many unique values for basic OHE
columns_to_drop_common = ['Make', 'Model', 'VIN']
X_baseline = X.drop(columns=columns_to_drop_common, errors='ignore')
numerical_features_baseline = [col for col in numerical_features if col not in columns_to_drop_common]
categorical_features_baseline = [col for col in categorical_features if col not in columns_to_drop_common]


# --- Step 1: Baseline Model (from Task 1 Simulation) ---
print("\n--- Baseline Model Performance (Task 1 Simulation) ---")

# Preprocessor for baseline model
numerical_transformer_baseline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer_baseline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # sparse_output=False for dense array
])

preprocessor_baseline = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer_baseline, numerical_features_baseline),
        ('cat', categorical_transformer_baseline, categorical_features_baseline)
    ],
    remainder='drop' # Drop columns not specified (e.g., 'Make', 'Model' if not already dropped)
)

# Baseline model: RandomForestRegressor with default parameters
baseline_model = Pipeline(steps=[('preprocessor', preprocessor_baseline),
                                 ('regressor', RandomForestRegressor(random_state=42))])

# Split data for baseline
X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline = train_test_split(X_baseline, y, test_size=0.2, random_state=42)

# Train baseline model
baseline_model.fit(X_train_baseline, y_train_baseline)

# Evaluate baseline model
y_pred_baseline = baseline_model.predict(X_test_baseline)
r2_baseline = r2_score(y_test_baseline, y_pred_baseline)
mae_baseline = mean_absolute_error(y_test_baseline, y_pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_test_baseline, y_pred_baseline))

print(f"Baseline R-squared: {r2_baseline:.4f}")
print(f"Baseline MAE: {mae_baseline:.2f}")
print(f"Baseline RMSE: {rmse_baseline:.2f}")


# --- Step 2: Feature Engineering and Tuned Model ---
print("\n--- Feature Engineering and Tuned Model ---")

# Create new engineered features
def create_engineered_features(data_frame):
    df_eng = data_frame.copy()

    # Feature 1: Power-to-Weight Ratio (using 'hp' and 'weightlbs')
    if 'hp' in df_eng.columns and ' weightlbs' in df_eng.columns:
        # Ensure 'weightlbs' is numeric. It might be object type due to spaces or non-numeric chars.
        df_eng[' weightlbs'] = pd.to_numeric(df_eng[' weightlbs'], errors='coerce')
        df_eng['Power_to_Weight_Ratio'] = df_eng['hp'] / df_eng[' weightlbs'].replace(0, np.nan)
        df_eng['Power_to_Weight_Ratio'].fillna(df_eng['Power_to_Weight_Ratio'].mean(), inplace=True)
    else:
        print("Warning: 'hp' or ' weightlbs' not found for 'Power_to_Weight_Ratio' feature. Defaulting to 0.")
        df_eng['Power_to_Weight_Ratio'] = 0 # Fallback

    # Feature 2: Engine Size Categories (binning using ' cylinders' or 'hp')
    if ' cylinders' in df_eng.columns:
        # Fill NaN before binning, otherwise pd.cut might drop them
        df_eng[' cylinders'].fillna(df_eng[' cylinders'].median(), inplace=True)
        bins = [-np.inf, 4, 6, 8, np.inf]
        labels = ['Small_Engine', 'Medium_Engine', 'Large_Engine', 'V_Engine']
        df_eng['Engine_Category'] = pd.cut(df_eng[' cylinders'], bins=bins, labels=labels, right=False, include_lowest=True)
        df_eng['Engine_Category'] = df_eng['Engine_Category'].astype(object).fillna('Unknown') # Convert to object to handle 'Unknown'
    elif 'hp' in df_eng.columns:
        df_eng['hp'].fillna(df_eng['hp'].median(), inplace=True)
        bins = [-np.inf, 150, 250, 350, np.inf]
        labels = ['Low_HP', 'Medium_HP', 'High_HP', 'Very_High_HP']
        df_eng['Engine_Category'] = pd.cut(df_eng['hp'], bins=bins, labels=labels, right=False, include_lowest=True)
        df_eng['Engine_Category'] = df_eng['Engine_Category'].astype(object).fillna('Unknown')
    else:
        print("Warning: Neither ' cylinders' nor 'hp' found for 'Engine_Category' feature. Defaulting to 'Unknown'.")
        df_eng['Engine_Category'] = 'Unknown' # Fallback

    # Feature 3: Car Age (using ' year')
    if ' year' in df_eng.columns:
        df_eng['Car_Age'] = 2023 - df_eng[' year']
        df_eng['Car_Age'].fillna(df_eng['Car_Age'].mean(), inplace=True)
    else:
        print("Warning: ' year' column not found for 'Car_Age' feature. Defaulting to 0.")
        df_eng['Car_Age'] = 0 # Fallback

    return df_eng

# Apply feature engineering to the original X data
X_engineered = create_engineered_features(X)

# Drop original columns that were used to create new features or are now redundant
# This helps prevent multicollinearity and keeps the feature set cleaner.
# Note: ' cubicinches' and ' weightlbs' are initially object types, so they need to be handled properly
# 'hp' is used for ratio but also has its own importance
# ' year' is replaced by 'Car_Age'

columns_to_drop_engineered = [' year'] # Car_Age replaces ' year' conceptually

X_engineered_processed = X_engineered.drop(columns=columns_to_drop_engineered, errors='ignore')
X_engineered_processed = X_engineered_processed.drop(columns=columns_to_drop_common, errors='ignore') # Ensure common drop columns are also out

# Update feature lists for the tuned model
numerical_features_tuned = X_engineered_processed.select_dtypes(include=np.number).columns.tolist()
categorical_features_tuned = X_engineered_processed.select_dtypes(include='object').columns.tolist()

# Preprocessing for the tuned model
numerical_transformer_tuned = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer_tuned = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_tuned = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer_tuned, numerical_features_tuned),
        ('cat', categorical_transformer_tuned, categorical_features_tuned)
    ],
    remainder='drop'
)

# Define the model to tune (GradientBoostingRegressor)
tuned_regressor = GradientBoostingRegressor(random_state=42)

# Create a pipeline for the tuned model
pipeline_tuned = Pipeline(steps=[('preprocessor', preprocessor_tuned),
                                  ('regressor', tuned_regressor)])

# Define parameter grid for GridSearchCV
param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__max_depth': [3, 5],
    'regressor__min_samples_split': [2, 4],
    'regressor__min_samples_leaf': [1, 2]
}

# Split data for tuning
X_train_tuned, X_test_tuned, y_train_tuned, y_test_tuned = train_test_split(X_engineered_processed, y, test_size=0.2, random_state=42)

# Perform GridSearchCV
print("\nPerforming GridSearchCV... This might take a while.")
# Using n_jobs=-1 to use all available cores for parallel processing
grid_search = GridSearchCV(pipeline_tuned, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train_tuned, y_train_tuned)

print(f"\nBest parameters found: {grid_search.best_params_}")

# Evaluate the best tuned model
best_tuned_model = grid_search.best_estimator_
y_pred_tuned = best_tuned_model.predict(X_test_tuned)

r2_tuned = r2_score(y_test_tuned, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test_tuned, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test_tuned, y_pred_tuned))

print(f"Tuned Model R-squared: {r2_tuned:.4f}")
print(f"Tuned Model MAE: {mae_tuned:.2f}")
print(f"Tuned Model RMSE: {rmse_tuned:.2f}")

# --- Step 3: Compare Tuned Model vs. Baseline ---
print("\n--- Model Comparison ---")

# Calculate improvements
r2_improvement = r2_tuned - r2_baseline
mae_improvement = mae_baseline - mae_tuned  # Lower MAE is better
rmse_improvement = rmse_baseline - rmse_tuned # Lower RMSE is better

print(f"Baseline R-squared: {r2_baseline:.4f}")
print(f"Tuned R-squared:    {r2_tuned:.4f} (Improvement: {r2_improvement:.4f})")
print(f"Baseline MAE: {mae_baseline:.2f}")
print(f"Tuned MAE:    {mae_tuned:.2f} (Improvement: {mae_improvement:.2f})")
print(f"Baseline RMSE: {rmse_baseline:.2f}")
print(f"Tuned RMSE:    {rmse_tuned:.2f} (Improvement: {rmse_improvement:.2f})")

if r2_improvement > 0:
    print("\nConclusion: The tuned model with engineered features shows an improvement in R-squared over the baseline model.")
elif mae_improvement > 0 and rmse_improvement > 0:
    print("\nConclusion: The tuned model with engineered features shows an improvement in MAE and RMSE over the baseline model, even if R-squared did not increase significantly.")
else:
    print("\nConclusion: The tuned model with engineered features did not significantly outperform the baseline model based on the evaluated metrics.")


Dataset loaded successfully.

Initial DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           261 non-null    float64
 1    cylinders    261 non-null    int64  
 2    cubicinches  261 non-null    object 
 3    hp           261 non-null    int64  
 4    weightlbs    261 non-null    object 
 5    time-to-60   261 non-null    int64  
 6    year         261 non-null    int64  
 7    brand        261 non-null    object 
dtypes: float64(1), int64(4), object(3)
memory usage: 16.4+ KB

Initial DataFrame Head:
    mpg   cylinders  cubicinches   hp  weightlbs   time-to-60   year     brand
0  14.0           8          350  165       4209           12   1972       US.
1  31.9           4           89   71       1925           14   1980   Europe.
2  17.0           8          302  140       3449           11   1971       US.
3  15.0 

/tmp/ipykernel_4227/2203185212.py:123: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_eng[' cylinders'].fillna(df_eng[' cylinders'].median(), inplace=True)
/tmp/ipykernel_4227/2203185212.py:141: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method


Best parameters found: {'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 4, 'regressor__n_estimators': 100}
Tuned Model R-squared: 0.8823
Tuned Model MAE: 1.99
Tuned Model RMSE: 2.71

--- Model Comparison ---
Baseline R-squared: 0.8732
Tuned R-squared:    0.8823 (Improvement: 0.0091)
Baseline MAE: 2.01
Tuned MAE:    1.99 (Improvement: 0.02)
Baseline RMSE: 2.81
Tuned RMSE:    2.71 (Improvement: 0.10)

Conclusion: The tuned model with engineered features shows an improvement in R-squared over the baseline model.
